### Instalación de requerimientos e importación de bibliotecas

In [1]:
%pip install -r requirements.txt
import os
import gdown
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import contextily as ctx
import folium
import re

import polars as pl
import gc

from IPython.display import display


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Descarga de archivos y operaciones de carpetas

In [2]:
# Función para descarga de datasets con gdown

# URL de Google Drive
url = "https://drive.google.com/uc?id=ID_DEL_ARCHIVO"

def descargar_archivos_desde_drive(lista_ids):
    """
    Descarga varios archivos de Google Drive a partir de una lista de IDs.

    Args:
        lista_ids (list): Una lista de strings con los IDs de los archivos de Drive.
    """
    for file_id in lista_ids:
        # Construimos la URL de descarga directa
        url = f'https://drive.google.com/uc?id={file_id}'

        print(f"\nIniciando descarga del ID: {file_id}")

        try:
            # gdown detecta automáticamente el nombre original del archivo
            gdown.download(url, quiet=False)
        except Exception as e:
            print(f"No se pudo descargar el archivo {file_id}. Error: {e}")

In [3]:
# Definición de carpeta de destino para CSV operación del árbol de directorios

# Definir la carpeta y el archivo de destino
output_dir = "./content/csv"

# Crear la carpeta si no existe
os.makedirs(output_dir, exist_ok=True)

# Se cambia a la carpeta de descaga de CSV
os.chdir(output_dir)

In [4]:
# Carga de CSV desde Google Drive
 
'''

lista_ids_datasets = [
    '1sWI3jP6f9VDJ-IE1EI1lkc8rkxooP3b7', # lineas-de-subte
    '1V6Cjhf2QU_gcig6HT5EvXhk0n2Egerqr', # estaciones-accesibles
    '1CyWPBgfAYRBcYvQO7U7cRlAoQbGWoosP', # historico_2014
    '1g7LpNJFqNcqDmgaGc81MMgV3VfrD6Ah6', # historico_2015
    '1QqOb3oLoMs014d2YBYw4_e8ww4jo-JX1', # historico_2016
    '1G7noINplTWyt2g9xRrS7l0BKgFOW05hv', # historico_2017
    '11WgJxZsC4zUURSlCUBEQKXCQK5RLkRNZ', # historico_2018
    '1DVgvubSgYCTPQCfA4zj5eiH_ni136o9A', # historico_2019
    '1hlfAVsJS20InzvIkUXT4t5_nOgd2m1NW', # historico_2020
    '1hW4qHioTzrXlDnBfWextpMQpar0kmL03', # historico_2021
    '1yETNbct23DLqYoN7ti6hNV3RdKErwMYI', # registro-historico-del-precio-del-boleto
    '1mY28zAPaI79Pt-OLoNSAhIxnCtflpmKU', # registro-historico-del-precio-del-boleto.xlsx
    '1PEAW6Vik2k-J7k9C6gxNtl_2df636Q41'  # viajes_anual
    ]

descargar_archivos_desde_drive(lista_ids_datasets)
'''

"\n\nlista_ids_datasets = [\n    '1sWI3jP6f9VDJ-IE1EI1lkc8rkxooP3b7', # lineas-de-subte\n    '1V6Cjhf2QU_gcig6HT5EvXhk0n2Egerqr', # estaciones-accesibles\n    '1CyWPBgfAYRBcYvQO7U7cRlAoQbGWoosP', # historico_2014\n    '1g7LpNJFqNcqDmgaGc81MMgV3VfrD6Ah6', # historico_2015\n    '1QqOb3oLoMs014d2YBYw4_e8ww4jo-JX1', # historico_2016\n    '1G7noINplTWyt2g9xRrS7l0BKgFOW05hv', # historico_2017\n    '11WgJxZsC4zUURSlCUBEQKXCQK5RLkRNZ', # historico_2018\n    '1DVgvubSgYCTPQCfA4zj5eiH_ni136o9A', # historico_2019\n    '1hlfAVsJS20InzvIkUXT4t5_nOgd2m1NW', # historico_2020\n    '1hW4qHioTzrXlDnBfWextpMQpar0kmL03', # historico_2021\n    '1yETNbct23DLqYoN7ti6hNV3RdKErwMYI', # registro-historico-del-precio-del-boleto\n    '1mY28zAPaI79Pt-OLoNSAhIxnCtflpmKU', # registro-historico-del-precio-del-boleto.xlsx\n    '1PEAW6Vik2k-J7k9C6gxNtl_2df636Q41'  # viajes_anual\n    ]\n\ndescargar_archivos_desde_drive(lista_ids_datasets)\n"

### Carga de datasets accesorios y configuración global

Los datasets accesorios (líneas, estaciones, precios, viajes) son chicos y se mantienen en memoria todo el tiempo. Se cargan una sola vez.

In [5]:
# Configuración global y carga de accesorios

null_values = ["", " ", "null", "NULL", "NaN", "NA", "N/A", "-"]

# Config de visualización de Polars (no mutila strings largos)
pl.Config.set_tbl_rows(100)
pl.Config.set_fmt_str_lengths(100)

lineas_subte_df = pl.read_csv('lineas-de-subte.csv', encoding='latin-1', null_values=null_values)
estaciones_accesibles_df = pl.read_csv('estaciones-accesibles.csv', encoding='latin-1', null_values=null_values)
registro_historico_del_precio_del_boleto_df = pl.read_csv('registro-historico-del-precio-del-boleto.csv', encoding='latin-1', null_values=null_values)
registro_historico_del_precio_del_boleto_excel_df = pl.read_excel('registro-historico-del-precio-del-boleto.xlsx')
viajes_anual_df = pl.read_csv('viajes_anual.csv', encoding='latin-1', null_values=null_values)

print("Accesorios cargados:")
for n in ["lineas_subte_df","estaciones_accesibles_df","registro_historico_del_precio_del_boleto_df","viajes_anual_df"]:
    print(f"  • {n}: {globals()[n].shape}")


Accesorios cargados:
  • lineas_subte_df: (82, 3)
  • estaciones_accesibles_df: (93, 6)
  • registro_historico_del_precio_del_boleto_df: (304, 4)
  • viajes_anual_df: (48, 3)


### Exploración de estaciones accesibles sin escaleras mecánicas ni ascensores

In [6]:
# Exploración de estaciones accesibles sin escaleras mecánicas ni ascensores

# Se filtra usando .filter() y expresiones pl.col
resultado_df = estaciones_accesibles_df.filter(
    (pl.col("escaleras_mecanicas") == 0) & (pl.col("ascensores") == 0)
)

# Lo mostrás directamente (acordate que no hace falta display si es lo último)
resultado_df

long,lat,linea,estacion,escaleras_mecanicas,ascensores
f64,f64,str,str,i64,i64


### Observaciones

Se observa que no hay estaciones listadas como estaciones accesibles que posean 0 escaleras mecánicas y 0 ascensores, lo cual es correcto.

## Pipeline de procesamiento por año (carga → curación → exportación → liberación)

A diferencia del enfoque anterior (que cargaba los 8 años en RAM simultáneamente), acá cada año se procesa de forma **aislada**: se carga, se cura, se diagnostica, se exporta a Parquet y se libera la memoria antes de pasar al siguiente. En ningún momento conviven dos años en RAM.

Cada etapa de curación es una función pura (recibe un `DataFrame`, devuelve un `DataFrame`). La función maestra `procesar_anio()` las orquesta e imprime los diagnósticos mientras el año está en memoria.

### Definiciones y funciones de etapa

In [7]:
# Constantes del pipeline

COLUMNAS_DESEADAS = [
    "FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION", "PAX_PAGO",
    "PAX_PAGOS", "PAX_PASES_PAGOS",
    "PAX_FREQ", "PAX_FRANQ", "PAX_TOTAL", "TOTAL"
]

# Carpeta de salida de los parquet (un nivel arriba del cwd ./content/csv -> ./content/parquet)
CARPETA_PARQUET = os.path.abspath(os.path.join(os.getcwd(), "..", "parquet"))
os.makedirs(CARPETA_PARQUET, exist_ok=True)
print(f"Los parquet se exportarán a: {CARPETA_PARQUET}")


Los parquet se exportarán a: /home/seijaku/Desktop/metodologia/content/parquet


In [8]:
# --- ETAPA 1: Carga selectiva de columnas ---

def cargar_anio(anio: int) -> pl.DataFrame | None:
    """Carga un CSV anual trayendo solo las columnas de interés.
    Maneja los casos especiales: 2021 (separador ';') y 2016/2017 (encoding lossy)."""
    ruta_csv = f"historico_{anio}.csv"
    separador = ";" if anio == 2021 else ","
    encoding = "utf8-lossy" if anio in (2016, 2017) else "utf8"

    try:
        lf = pl.scan_csv(
            ruta_csv,
            null_values=null_values,
            infer_schema_length=0,   # todo como String; casteamos nosotros después
            separator=separador,
            encoding=encoding,
        )
        columnas_reales = lf.collect_schema().names()
        columnas_a_importar = [c for c in columnas_reales if c.upper() in COLUMNAS_DESEADAS]
        df = lf.select(columnas_a_importar).collect()
        print(f"  [carga] {ruta_csv} | sep='{separador}' enc='{encoding}' | "
              f"cols={columnas_a_importar} | shape={df.shape}")
        return df
    except FileNotFoundError:
        print(f"  [carga] No se encontró {ruta_csv} — se omite el año {anio}.")
        return None


In [9]:
# --- ETAPA 2: Estandarización de nomenclaturas y mayúsculas ---

def estandarizar_columnas(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """Unifica nombres: PAX_FREQ/PAX_FRANQUICIAS -> PAX_FRANQ, TOTAL -> PAX_TOTAL,
    PAX_PAGO -> PAX_PAGOS (caso 2014), y finalmente todo a MAYÚSCULAS."""
    renombrado = {}
    cols_upper = [c.upper() for c in df.columns]

    for col in df.columns:
        cu = col.upper()
        if cu in ("PAX_FREQ", "PAX_FRANQUICIAS"):
            renombrado[col] = "PAX_FRANQ"
        # TOTAL -> PAX_TOTAL solo si no existe ya PAX_TOTAL (corrige el bug de indentación original)
        elif cu == "TOTAL" and "PAX_TOTAL" not in cols_upper:
            renombrado[col] = "PAX_TOTAL"
        elif cu == "PAX_PAGO":   # caso 2014: singular -> plural
            renombrado[col] = "PAX_PAGOS"

    if renombrado:
        df = df.rename(renombrado)
        print(f"  [nomenclatura] {anio}: {renombrado}")

    # Todo a mayúsculas (idempotente)
    df = df.rename(lambda c: c.upper())
    return df


In [10]:
# --- ETAPA 3: Tipado de fechas + imputación de PAX con 0 ---

def tipar_fechas_y_pax(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """Castea columnas PAX a Float64 (imputando nulos con 0) y FECHA a pl.Date
    con parseo elástico (ISO / latino / yanki)."""
    col_fecha = "FECHA"
    columnas_pax = [c for c in df.columns if "PAX_" in c or c == "TOTAL"]

    # Filtramos filas con FECHA vacía/nula
    df = df.filter(
        (pl.col(col_fecha).str.strip_chars() != "") & (pl.col(col_fecha).is_not_null())
    )

    # PAX -> float, limpiando espacios, e imputando 0
    operaciones = []
    for c in columnas_pax:
        if df.schema[c] == pl.String:
            expr = pl.col(c).str.replace_all(" ", "").cast(pl.Float64, strict=False)
        else:
            expr = pl.col(c)
        operaciones.append(expr.fill_null(0))
    if operaciones:
        df = df.with_columns(operaciones)

    # FECHA -> Date con coalesce de formatos
    if df.schema[col_fecha] == pl.String:
        df = df.with_columns(
            pl.coalesce([
                pl.col(col_fecha).str.to_date(format="%Y-%m-%d", strict=False),
                pl.col(col_fecha).str.to_date(format="%d/%m/%Y", strict=False),
                pl.col(col_fecha).str.to_date(format="%m/%d/%Y", strict=False),
            ]).alias(col_fecha)
        )
    nulos_fecha = df[col_fecha].is_null().sum()
    print(f"  [tipado fecha/pax] {anio}: pax imputadas={columnas_pax} | nulos FECHA restantes={nulos_fecha}")
    return df


In [11]:
# --- ETAPA 4: Tipado de horas (DESDE/HASTA -> pl.Time) ---

def tipar_horas(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """Convierte DESDE y HASTA a pl.Time, tolerando formato 'HH:MM' (le agrega ':00')."""
    cols_hora = [c for c in df.columns if c in ("DESDE", "HASTA")]
    transformaciones = {}
    for c in cols_hora:
        if df.schema[c] == pl.String:
            expr = pl.col(c).str.strip_chars()
            expr = pl.when(expr.str.len_chars() == 5).then(expr + ":00").otherwise(expr)
            transformaciones[c] = expr.str.to_time(format="%H:%M:%S", strict=False)
    if transformaciones:
        df = df.with_columns(**transformaciones)
        print(f"  [tipado hora] {anio}: convertidas {list(transformaciones.keys())} a pl.Time")
    return df


In [12]:
# --- ETAPA 5: Diagnóstico de nulos (solo reporta) ---

def diagnosticar_nulos(df: pl.DataFrame, anio: int) -> None:
    reporte = df.select([
        pl.all().null_count().name.suffix("_nulos"),
        (pl.col(pl.String).str.strip_chars() == "").sum().name.suffix("_vacios"),
    ]).transpose(include_header=True, header_name="Columna_Métrica", column_names=["Cantidad"])
    print(f"  [nulos] HISTÓRICO {anio}:")
    print(reporte)


In [13]:
# --- ETAPA 6: Correcciones específicas por año ---

def correccion_2018_bonifacio(df: pl.DataFrame) -> pl.DataFrame:
    """2018: la estación 'Taller Bonifacio' aparece con LINEA nula y PAX_TOTAL=0.
    Eliminamos esos registros espurios."""
    bonifacio = df.filter(pl.col("ESTACION") == "Taller Bonifacio")
    if bonifacio.height:
        print(f"  [corr 2018] Taller Bonifacio: {bonifacio.height} registros hallados, "
              f"se eliminan los de PAX_TOTAL=0")
    return df.filter(
        ~((pl.col("ESTACION") == "Taller Bonifacio") & (pl.col("PAX_TOTAL") == 0.0))
    )

def consolidar_duplicados_clave(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """2020/2021: filas idénticas y duplicados por clave espacio-temporal.
    Se eliminan idénticas y se suman las métricas por clave."""
    antes = df.height
    df = df.unique()
    df = (
        df.group_by(["FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION"])
          .agg([
              pl.col("PAX_PAGOS").sum(),
              pl.col("PAX_PASES_PAGOS").sum(),
              pl.col("PAX_FRANQ").sum(),
              pl.col("PAX_TOTAL").sum(),
          ])
    )
    print(f"  [consolidación] {anio}: {antes:,} -> {df.height:,} filas "
          f"(reducción {antes - df.height:,})")
    return df

def aplicar_correcciones_especificas(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    if anio == 2018:
        df = correccion_2018_bonifacio(df)
    if anio in (2020, 2021):
        df = consolidar_duplicados_clave(df, anio)
    return df


In [14]:
# --- ETAPA 7: Diagnóstico de duplicados (solo reporta) ---

def diagnosticar_duplicados(df: pl.DataFrame, anio: int) -> None:
    total = df.height
    dup_exactos = total - df.unique().height
    cols_clave = [c for c in df.columns
                  if c in ("FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION")]
    dup_clave = total - df.unique(subset=cols_clave).height
    print(f"  [duplicados] {anio}: total={total:,} | idénticos={dup_exactos:,} | "
          f"por clave={dup_clave:,}")


In [15]:
# --- ETAPA 8: Diagnóstico de outliers (solo reporta) ---

def diagnosticar_outliers(df: pl.DataFrame, anio: int) -> None:
    # Numéricos
    col_total = next((c for c in df.columns if "PAX_TOTAL" in c), None)
    if col_total:
        s = df.select([
            pl.col(col_total).min().alias("min"),
            pl.col(col_total).max().alias("max"),
            pl.col(col_total).mean().alias("media"),
            pl.col(col_total).std().alias("std"),
            (pl.col(col_total) < 0).sum().alias("negativos"),
        ])
        print(f"  [outliers num] {anio} ({col_total}): "
              f"min={s['min'][0]} max={s['max'][0]:,} media={s['media'][0]:.2f} "
              f"std={s['std'][0]:.2f} negativos={s['negativos'][0]}")
        if s['max'][0] is not None and s['max'][0] > 15000:
            print(f"      ALERTA: máximo físicamente improbable para 15 min.")
        if s['negativos'][0] > 0:
            print(f"      ALERTA: valores negativos presentes.")

    # Temporales
    dft = df.filter(pl.col("FECHA").is_not_null())
    fuera_anio = dft.filter(pl.col("FECHA").dt.year() != anio).height
    horas_rotas = dft.filter(pl.col("DESDE").is_null()).height
    print(f"  [outliers temp] {anio}: fechas fuera de año={fuera_anio} | horas DESDE nulas={horas_rotas}")
    if fuera_anio > 0:
        anios_det = dft.filter(pl.col("FECHA").dt.year() != anio)\
                       .select(pl.col("FECHA").dt.year()).unique().to_series().to_list()
        print(f"      ALERTA: años mezclados detectados: {anios_det}")


In [16]:
# --- ETAPA 9: Estandarización de LINEA ---

def estandarizar_linea(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """Extrae la letra de ramal (A-H) del final del string: 'LineaA'->'A', 'LINEA_A'->'A'."""
    df = df.with_columns(pl.col("LINEA").str.extract(r"([A-H])$", 1))
    unicos = df.select(pl.col("LINEA")).unique().sort("LINEA")["LINEA"].to_list()
    print(f"  [linea] {anio}: ramales -> {unicos}")
    return df


### Función maestra y ejecución del pipeline

In [17]:
# --- FUNCIÓN MAESTRA: procesa un año de punta a punta y libera la memoria ---

def procesar_anio(anio: int, verbose_nulos: bool = False) -> bool:
    """Carga, cura, diagnostica y exporta a Parquet un único año.
    Devuelve True si se procesó, False si el archivo no existía.
    Al terminar, el DataFrame queda fuera de scope y se libera con gc.collect()."""
    print(f"\n{'='*70}\nPROCESANDO AÑO {anio}\n{'='*70}")

    df = cargar_anio(anio)
    if df is None:
        return False

    # Curación
    df = estandarizar_columnas(df, anio)
    df = tipar_fechas_y_pax(df, anio)
    df = tipar_horas(df, anio)

    # Diagnóstico de nulos (el detalle de la tabla solo si se pide)
    if verbose_nulos:
        diagnosticar_nulos(df, anio)

    # Correcciones específicas (2018 Bonifacio, 2020/2021 consolidación)
    df = aplicar_correcciones_especificas(df, anio)

    # Diagnósticos finales
    diagnosticar_duplicados(df, anio)
    diagnosticar_outliers(df, anio)

    # Estandarización de LINEA (después de eliminar nulos de Bonifacio)
    df = estandarizar_linea(df, anio)

    # Exportación
    ruta_parquet = os.path.join(CARPETA_PARQUET, f"historico_{anio}.parquet")
    df.write_parquet(ruta_parquet, compression="snappy")
    print(f"  [export] {ruta_parquet} ({df.height:,} filas)")

    # Liberación explícita
    del df
    gc.collect()
    print(f"  [memoria] año {anio} liberado.")
    return True


In [18]:
# --- EJECUCIÓN: un año por vez, sin acumular en RAM ---

procesados = []
for anio in range(2014, 2022):
    if procesar_anio(anio, verbose_nulos=False):
        procesados.append(anio)
    gc.collect()  # seguro adicional entre años

print(f"\n{'='*70}")
print(f"Pipeline finalizado. Años procesados y exportados a Parquet: {procesados}")
print(f"{'='*70}")



PROCESANDO AÑO 2014
  [carga] historico_2014.csv | sep=',' enc='utf8' | cols=['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGO', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL'] | shape=(10857244, 10)
  [nomenclatura] 2014: {'PAX_PAGO': 'PAX_PAGOS'}
  [tipado fecha/pax] 2014: pax imputadas=['PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL'] | nulos FECHA restantes=0
  [tipado hora] 2014: convertidas ['DESDE', 'HASTA'] a pl.Time
  [duplicados] 2014: total=10,857,244 | idénticos=0 | por clave=0
  [outliers num] 2014 (PAX_TOTAL): min=0.0 max=14,538.0 media=23.50 std=27.73 negativos=0
  [outliers temp] 2014: fechas fuera de año=0 | horas DESDE nulas=0
  [linea] 2014: ramales -> ['A', 'B', 'C', 'D', 'E', 'H']
  [export] /home/seijaku/Desktop/metodologia/content/parquet/historico_2014.parquet (10,857,244 filas)
  [memoria] año 2014 liberado.

PROCESANDO AÑO 2015
  [carga] historico_2015.csv | sep=',' enc='utf8' | cols=['fecha', 'desde', 'hasta', 'linea', 'molinete', 

### Activación del motor Lazy sobre los Parquet unificados

Una vez exportados todos los años, se mapean de forma perezosa con un único `scan_parquet` y comodín. Esto no carga nada a RAM: arma el plan de ejecución que se materializa recién al hacer `.collect()`.

In [19]:
# Conexión Lazy unificada sobre todos los parquet

ruta_busqueda = os.path.join(CARPETA_PARQUET, "historico_*.parquet")
subte_lazy_df = pl.scan_parquet(ruta_busqueda)

print("Dataset unificado conectado de forma Lazy.")
print(f"• Esquema virtual: {subte_lazy_df.collect_schema().names()}")
print(f"• Tipo de objeto: {type(subte_lazy_df)}")


Dataset unificado conectado de forma Lazy.
• Esquema virtual: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'MOLINETE', 'ESTACION', 'PAX_PAGOS', 'PAX_PASES_PAGOS', 'PAX_FRANQ', 'PAX_TOTAL']
• Tipo de objeto: <class 'polars.lazyframe.frame.LazyFrame'>


https://github.com/AleLoredo/UGR-metodologia/blob/eze/UGR_Metodologia_TP1_subte.ipynb